# Midas Design Guide — Prestressed Concrete Bridge Load Rating

**Companion notebook** for the corresponding chapter of the MIDAS training
manual *Design Guide for midas Civil — AASHTO LRFD*. The guide itself is
proprietary and is **not reproduced here** — this notebook contains only
original code and AASHTO LRFD / MBE article citations.

**How this notebook is used.** Work through the design process with this
notebook alongside: each step carries its AASHTO background in original
words, a live Python environment for exploratory checks and quick
validation, a direct interface to Midas Civil through its API, and
customization to ODOT's design process (PSID / PSBD standard products,
ODOT materials and vehicles).
Everything the guide has you do by hand in the Midas UI — coordinate entry,
side math — is demonstrated here as runnable code a designer can follow,
rerun, and modify. Where a number must come from the
Midas model itself, it is either pulled live over the Civil NX JSON API
(the verified result tables run in place) or recorded from the companion
six-beam model with its provenance noted; the remaining `TODO(midas-api)`
markers await the PSC design module and construction-stage analysis. Hand-entered values (from a standard
drawing or a hand calc) are compared via the `check()` harness below.

**Scope of this chapter:**

1. Rating vehicle set and MBE 6A load/resistance factors
2. Bridge inputs (usually the Chapter 1 girder) and capacities at rating limit states
3. Force effects per vehicle from the Midas moving-load analysis (API)
4. Design load rating — HL-93 inventory / operating (strength + Service III)
5. Legal load rating with ADTT-based factors
6. Permit load rating
7. Posting evaluation and cross-check vs Midas rating tables (API)


In [ ]:
import math
import pandas as pd

# civilpy is installed editable into this env (pip install -e .)
from civilpy.structural.midas import MidasCivil, parse_result_table, envelope
from civilpy.structural.aashto.lrfd import (
    concrete, prestressed, steel, composite, distribution, lrfr, creep_shrinkage,
)

# --- Midas Civil NX connection -------------------------------------------------
# The API is not running on this machine right now. Everything below that needs
# the live model is guarded by MIDAS_ONLINE and marked TODO(midas-api).
try:
    midas = MidasCivil()
    MIDAS_ONLINE = midas.ping()
except Exception:
    midas, MIDAS_ONLINE = None, False
print("Midas Civil NX online:", MIDAS_ONLINE)

In [ ]:
# --- Validation harness --------------------------------------------------------
# Every comparison in this notebook goes through check() so the end-of-notebook
# summary shows guide value vs civilpy value side by side.
RESULTS = []

def check(label, guide_value, civilpy_value, tol=0.01, unit=""):
    """Compare a guide-reported value against the civilpy-computed one.

    tol is relative (1% default) — the guide rounds intermediate values, so
    small drift is expected; flag anything beyond tol for investigation.
    """
    if guide_value is None or civilpy_value is None:
        status = "PENDING"
        diff = None
    else:
        diff = abs(civilpy_value - guide_value) / (abs(guide_value) or 1.0)
        status = "OK" if diff <= tol else "MISMATCH"
    RESULTS.append({"check": label, "guide": guide_value, "civilpy": civilpy_value,
                    "rel diff": diff, "unit": unit, "status": status})
    print(f"[{status}] {label}: guide={guide_value} civilpy={civilpy_value} {unit}")
    return status == "OK"

def summary():
    df = pd.DataFrame(RESULTS)
    if len(df):
        n_ok = (df.status == "OK").sum()
        print(f"{n_ok}/{len(df)} checks OK, "
              f"{(df.status == 'MISMATCH').sum()} mismatches, "
              f"{(df.status == 'PENDING').sum()} pending")
    return df

## 1. Rating setup — MBE Part 6A

Vehicles, condition/system factors, and the limit states rated. PSC design
load rating checks Strength I plus the Service III stress limit (inventory).

In [ ]:
# TODO(guide): record the guide's factor table here:
GUIDE = dict(
    phi_c=None,   # condition factor
    phi_s=None,   # system factor
    gamma_DC=1.25, gamma_DW=1.50,
    gamma_LL_inv=1.75, gamma_LL_op=1.35,
    adtt=None,
)
# Ohio legal vehicles already live in civilpy.structural.aashto.vehicles;
# TODO(guide): confirm which legal/permit vehicles the guide rates.
GUIDE

## 2. Capacities at the rating limit states

Reuse the Chapter 1 design notebook's capacity results (flexure, shear,
Service III stress margin). If rating a different example bridge, rebuild the
capacities here with the same civilpy calls.

In [ ]:
# TODO(guide): phi*Mn, phi*Vn at the rated sections, and the Service III
# allowable-stress capacity term for the stress-based rating.
pass

## 3. Force effects per rating vehicle

In [ ]:
if MIDAS_ONLINE:
    # Verified vs live Civil NX 2026-07-27: /post/TABLE selects by
    # TABLE_TYPE ("BEAMFORCE"); TABLE_NAME is just a label.
    try:
        resp = midas.result_table("Moving load envelopes",
                                  table_type="BEAMFORCE")
        display(pd.DataFrame(parse_result_table(resp)).head())
    except Exception as err:
        # a fresh/unanalyzed session (or a DB edit, or a pre-mode view
        # switch) clears results — analyze and rerun this cell
        print("no results in the session:", str(err)[-80:])
else:
    print("Midas offline — skipping Moving load envelopes (BEAMFORCE)")

## 4. Design load rating — HL-93

RF = (C − γDC·DC − γDW·DW) / (γLL·(LL+IM)) at Strength I, plus the
Service III stress-based rating for inventory.

In [ ]:
# TODO(guide):
# rf_inv = lrfr.rating_factor(capacity=..., dc=..., dw=..., ll_im=...,
#                             gamma_ll=1.75, phi_c=..., phi_s=...)
# rf_op  = lrfr.rating_factor(..., gamma_ll=1.35)
# check("RF inventory (flexure)", <guide>, rf_inv), etc.
pass

## 5. Legal load rating — MBE 6A.4.4

Generalized live-load factor from ADTT via `lrfr.legal_load_factor`.

In [ ]:
# TODO(guide):
# g_ll = lrfr.legal_load_factor(adtt=GUIDE["adtt"])
# RF per legal vehicle; posting check below if any RF < 1.0.
pass

## 6. Permit load rating — MBE 6A.4.5

In [ ]:
# TODO(guide): lrfr.permit_load_factor(...) — confirm permit type,
# escort conditions, and ADTT band against the guide's example.
pass

## 7. Posting and Midas rating-table cross-check

`lrfr.posting_load` for any legal RF < 1.0, then diff the full rating table
against what Civil NX reports.

In [ ]:
# TODO(midas-api): "Load Rating Result" is a PSC *design-module* table.
# Verified 2026-07-27: the design result tables (fps, c, Mcr, Av,req,
# FDL/AFDL columns) are NOT exposed through the known /post/TABLE surface —
# they need the PSC Design run configured in the Civil NX UI (design code,
# PSC design parameters, Section Manager rebar) and/or the official JSON
# manual's design TABLE_TYPE names. Analysis-side tables ARE verified:
# BEAMFORCE, BEAMSTRESSPSC (the ten-check-point stress table with
# Sig-Is(shear), Sig-Is(shear+torsion), Sig-Ps(Max/Min) columns), REACTIONG.
print("PENDING: Load Rating Result — needs the PSC Design module (see comment)")

## Validation summary

Every `check()` recorded above, in one table. `PENDING` rows are waiting on
either guide values (hand entry) or the Midas API coming back online.

In [ ]:
summary()